# Lab 5: Real-World Data Patterns - Solution

This notebook contains the complete solution for Lab 5. Use this to verify your implementation or if you get stuck.

## Step 1: Slowly Changing Dimensions (SCD Type 2)

In [ ]:
// Create customer dimension table
spark.sql("""
  CREATE TABLE iceberg.default.customer_dim (
    customer_key INT,
    customer_id STRING,
    name STRING,
    email STRING,
    address STRING,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    is_current BOOLEAN
  ) USING iceberg
  PARTITIONED BY (is_current)
  ORDER BY customer_id, valid_from
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert initial customer records (SCD Type 2)
spark.sql("""
  INSERT INTO iceberg.default.customer_dim VALUES
    (1, 'CUST001', 'Alice Johnson', 'alice@example.com', '123 Main St', 
     TIMESTAMP '2024-01-01 00:00:00', TIMESTAMP '9999-12-31 23:59:59', true),
    (2, 'CUST002', 'Bob Smith', 'bob@example.com', '456 Oak Ave', 
     TIMESTAMP '2024-01-01 00:00:00', TIMESTAMP '9999-12-31 23:59:59', true),
    (3, 'CUST003', 'Charlie Brown', 'charlie@example.com', '789 Pine Rd', 
     TIMESTAMP '2024-01-01 00:00:00', TIMESTAMP '9999-12-31 23:59:59', true)
""")

In [ ]:
// Simulate customer change (Alice moves)
val currentTime = "2024-01-15 00:00:00"

// First, expire old record
spark.sql(s"""
  UPDATE iceberg.default.customer_dim
  SET valid_to = TIMESTAMP '$currentTime',
      is_current = false
  WHERE customer_id = 'CUST001' AND is_current = true
""")

In [ ]:
// Then insert new record
spark.sql(s"""
  INSERT INTO iceberg.default.customer_dim VALUES
    (4, 'CUST001', 'Alice Johnson', 'alice.new@example.com', '456 New St', 
     TIMESTAMP '$currentTime', TIMESTAMP '9999-12-31 23:59:59', true)
""")

In [ ]:
// Query customer history
spark.sql("""
  SELECT * FROM iceberg.default.customer_dim
  WHERE customer_id = 'CUST001'
  ORDER BY valid_from
""").show()
println("✅ SCD Type 2 implementation successful!")

## Step 2: Upsert Operations

In [ ]:
// Create orders fact table for upsert operations
spark.sql("""
  CREATE TABLE iceberg.default.orders_fact (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DECIMAL(10,2),
    status STRING,
    last_updated TIMESTAMP
  ) USING iceberg
  PARTITIONED BY (order_date)
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.metadata.compression-codec'='gzip'
  )
""")

In [ ]:
// Insert initial orders
spark.sql("""
  INSERT INTO iceberg.default.orders_fact VALUES
    (1, 'CUST001', DATE '2024-01-01', 100.50, 'pending', TIMESTAMP '2024-01-01 00:00:00'),
    (2, 'CUST002', DATE '2024-01-01', 200.75, 'pending', TIMESTAMP '2024-01-01 00:00:00'),
    (3, 'CUST003', DATE '2024-01-02', 150.25, 'pending', TIMESTAMP '2024-01-02 00:00:00')
""")

In [ ]:
// Perform upsert using MERGE
// First, create staging table with new/updated data
spark.sql("""
  CREATE TABLE iceberg.default.orders_staging (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DECIMAL(10,2),
    status STRING,
    last_updated TIMESTAMP
  ) USING iceberg
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Load staging data (updates and inserts)
spark.sql("""
  INSERT INTO iceberg.default.orders_staging VALUES
    (2, 'CUST002', DATE '2024-01-01', 250.75, 'shipped', TIMESTAMP '2024-01-03 00:00:00'),  -- Update
    (4, 'CUST001', DATE '2024-01-03', 300.00, 'pending', TIMESTAMP '2024-01-03 00:00:00')   -- Insert
""")

In [ ]:
// Perform upsert using MERGE
spark.sql("""
  MERGE INTO iceberg.default.orders_fact AS target
  USING iceberg.default.orders_staging AS source
  ON target.order_id = source.order_id
  WHEN MATCHED THEN
    UPDATE SET
      customer_id = source.customer_id,
      amount = source.amount,
      status = source.status,
      last_updated = source.last_updated
  WHEN NOT MATCHED THEN
    INSERT *
""")

In [ ]:
// Verify upsert results
spark.sql("SELECT * FROM iceberg.default.orders_fact ORDER BY order_id").show()
println("✅ Upsert operations successful!")

## Step 3: Batch and Streaming Patterns

In [ ]:
// Create table for daily batch loading
spark.sql("""
  CREATE TABLE iceberg.default.daily_sales (
    sale_date DATE,
    product_id INT,
    quantity INT,
    revenue DECIMAL(10,2),
    region STRING,
    load_timestamp TIMESTAMP
  ) USING iceberg
  PARTITIONED BY (sale_date)
  ORDER BY product_id
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Simulate daily batch loads
val dates = Seq("2024-01-01", "2024-01-02", "2024-01-03")

for (date <- dates) {
  spark.sql(s"""
    INSERT INTO iceberg.default.daily_sales VALUES
    (DATE '$date', 101, 10, 1000.00, 'west', CURRENT_TIMESTAMP),
    (DATE '$date', 102, 15, 1500.00, 'east', CURRENT_TIMESTAMP),
    (DATE '$date', 103, 8, 800.00, 'west', CURRENT_TIMESTAMP)
  """)
  
  // Simulate daily compaction
  spark.sql("""
    CALL iceberg.system.rewrite_data_files(
      'iceberg.default.daily_sales',
      map(
        'min-input-files', '3',
        'target-size-bytes', str(128 * 1024 * 1024)
      )
    )
  """)
}

In [ ]:
// Query daily sales
spark.sql("""
  SELECT 
    sale_date, 
    SUM(quantity) as total_quantity,
    SUM(revenue) as total_revenue
  FROM iceberg.default.daily_sales
  GROUP BY sale_date
  ORDER BY sale_date
""").show()
println("✅ Batch loading with daily compaction successful!")

## ✅ Lab Completion Checklist

- [x] SCD Type 2 maintains complete customer history
- [x] Upsert operations using MERGE work correctly
- [x] Batch loading with daily compaction produces clean partitions

## 🎓 Key Concepts Learned

1. **SCD Type 2**: Maintaining historical dimension data with valid_from/valid_to
2. **Upsert Operations**: Using MERGE for insert-or-update logic
3. **Batch Patterns**: Daily data loading with compaction
4. **CDC Pattern**: Change data capture for synchronizing operational systems